# SkinCoach — Обучение модели v3.1
**Датасеты:** HAM10000 + Fitzpatrick17k (опционально)  
**Модель:** EfficientNet-B3 (N классов — определяется анализом данных)  
**Цель:** val_acc > 80%

## Как запустить на Kaggle:
1. Создать новый ноутбук на [kaggle.com/code](https://www.kaggle.com/code)
2. Settings → Accelerator → **GPU T4 x2**
3. Добавить датасет: **Skin cancer: HAM10000** (от surajghuwalewala)
4. Установить секреты в Kaggle → Add-ons → Secrets:
   - `HF_TOKEN` = ваш HuggingFace write token
   - `GITHUB_TOKEN` = ваш GitHub Personal Access Token (если репо приватный)
5. Нажать **Run All**

In [ ]:
# ── Шаг 0: Проверка GPU ──────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '⚠️ GPU не найден')

In [ ]:
# ── Шаг 1: Установка зависимостей ────────────────────────────────────────────
!pip install -q huggingface_hub torch torchvision pillow

In [ ]:
# ── Шаг 2: Клонируем/обновляем репозиторий SkinCoach ─────────────────────────
import os
import subprocess

# Пробуем взять GitHub токен из Kaggle Secrets (для приватного репо)
GITHUB_TOKEN = ''
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    GITHUB_TOKEN = secrets.get_secret('GITHUB_TOKEN')
    print('GitHub токен загружен из Kaggle Secrets')
except Exception:
    print('GITHUB_TOKEN не найден в Secrets — используем публичный доступ')

REPO_NAME = 'lipindanil02-eng/skincoach08.03'
if GITHUB_TOKEN:
    REPO_URL = f'https://{GITHUB_TOKEN}@github.com/{REPO_NAME}.git'
else:
    REPO_URL = f'https://github.com/{REPO_NAME}.git'

WORK_DIR = '/kaggle/working/skincoach'

if os.path.exists(WORK_DIR):
    print('Репозиторий уже есть — обновляем (git pull)...')
    result = subprocess.run(
        ['git', '-C', WORK_DIR, 'pull', '--ff-only'],
        capture_output=True, text=True
    )
    print(result.stdout or result.stderr)
    if result.returncode != 0:
        print('git pull не удался — пересклонируем')
        subprocess.run(['rm', '-rf', WORK_DIR])
        result = subprocess.run(['git', 'clone', REPO_URL, WORK_DIR], capture_output=True, text=True)
        print(result.stdout or result.stderr)
else:
    print('Клонируем репозиторий...')
    result = subprocess.run(['git', 'clone', REPO_URL, WORK_DIR], capture_output=True, text=True)
    print(result.stdout or result.stderr)

if not os.path.exists(os.path.join(WORK_DIR, 'prepare_dataset.py')):
    raise RuntimeError('❌ Клонирование не удалось! Добавьте GITHUB_TOKEN в Kaggle → Add-ons → Secrets')

os.chdir(WORK_DIR)
print('Рабочая директория:', os.getcwd())
print('Ключевые файлы:', [f for f in os.listdir('.') if f.endswith('.py')])

In [ ]:
# ── Шаг 3: Пути к датасетам ──────────────────────────────────────────────────
import os

def find_ham_dir():
    """Находим папку с HAM10000 — ищем CSV с нужными полями."""
    candidates = [
        '/kaggle/input/ham10000',
        '/kaggle/input/skin-cancer-mnist-ham10000',
    ]
    # Ищем автоматически в /kaggle/input/ рекурсивно
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'GroundTruth.csv' in files or 'HAM10000_metadata.csv' in files:
            return root
        if 'images' in dirs and any(f.endswith('.csv') for f in files):
            return root

    for c in candidates:
        if os.path.exists(c):
            return c
    return None

HAM_DIR  = find_ham_dir()
FITZ_DIR = '/kaggle/working/fitzpatrick17k'
OUT_DIR  = '/kaggle/working/dataset'
MIN_COUNT = 50

print(f'HAM_DIR  = {HAM_DIR}')
print(f'HAM10000 найден: {HAM_DIR is not None and os.path.exists(str(HAM_DIR))}')
if HAM_DIR and os.path.exists(HAM_DIR):
    print(f'Содержимое: {os.listdir(HAM_DIR)[:15]}')

In [ ]:
# ── Шаг 4: Скачиваем Fitzpatrick17k (опционально) ────────────────────────────
# Fitzpatrick17k требует запрос доступа у авторов
# Если нет доступа — обучение пойдёт только на HAM10000 (это ОК для MVP)

if not os.path.exists(FITZ_DIR):
    print('Клонируем метаданные Fitzpatrick17k...')
    os.system(f'git clone https://github.com/mattgroh/fitzpatrick17k.git {FITZ_DIR}')
    if os.path.exists(f'{FITZ_DIR}/fitzpatrick17k.csv'):
        print('Fitzpatrick17k CSV найден')
        print('Для полного обучения нужно скачать изображения с сайта авторов')
    else:
        print('Fitzpatrick17k CSV не найден — пропускаем')
else:
    print('Fitzpatrick17k уже скачан')

print(f'\nСтатус: Fitzpatrick17k = {os.path.exists(FITZ_DIR)}')

In [ ]:
# ── Шаг 5: Анализ датасетов ──────────────────────────────────────────────────
import subprocess, os

args = ['python', 'analyze_datasets.py']
if HAM_DIR and os.path.exists(HAM_DIR):
    args += ['--ham', HAM_DIR]
if os.path.exists(f'{FITZ_DIR}/fitzpatrick17k.csv'):
    args += ['--fitz', FITZ_DIR]
args += ['--threshold', str(MIN_COUNT)]

print('Запускаю:', ' '.join(args))
result = subprocess.run(args, capture_output=False, text=True)
if result.returncode != 0:
    print(f'⚠️ Выход с кодом {result.returncode}')

In [ ]:
# ── Шаг 6: Подготовка датасета ───────────────────────────────────────────────
import subprocess, os

args = ['python', 'prepare_dataset.py', '--out', OUT_DIR, '--min_count', str(MIN_COUNT)]
if HAM_DIR and os.path.exists(HAM_DIR):
    args += ['--ham', HAM_DIR]
if os.path.exists(f'{FITZ_DIR}/fitzpatrick17k.csv'):
    args += ['--fitz', FITZ_DIR]

print('Запускаю:', ' '.join(args))
result = subprocess.run(args, capture_output=False, text=True)
if result.returncode != 0:
    raise RuntimeError(f'❌ prepare_dataset.py завершился с кодом {result.returncode}')

In [ ]:
# ── Шаг 7: Статистика датасета ────────────────────────────────────────────────
import os

print('📊 Итоговый датасет:\n')
total_train, total_val = 0, 0
for cls in sorted(os.listdir(os.path.join(OUT_DIR, 'train'))):
    n_train = len(os.listdir(os.path.join(OUT_DIR, 'train', cls)))
    n_val   = len(os.listdir(os.path.join(OUT_DIR, 'val',   cls))) if os.path.exists(os.path.join(OUT_DIR, 'val', cls)) else 0
    total_train += n_train
    total_val   += n_val
    status = '✅' if n_train >= 100 else ('⚠️ ' if n_train > 0 else '❌')
    print(f'  {status} {cls:25s} train={n_train:5d}  val={n_val:4d}')

print(f'\n  Итого: train={total_train}, val={total_val}')

In [ ]:
# ── Шаг 8: ОБУЧЕНИЕ ───────────────────────────────────────────────────────────
import os, subprocess

HF_TOKEN = ''
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret('HF_TOKEN')
    print('HuggingFace токен загружен')
except Exception:
    print('HF_TOKEN не найден — модель не будет загружена на HuggingFace')
    print('   Добавьте токен: Kaggle → Add-ons → Secrets')

HF_REPO = 'danyil163/SCINCOACH'

args = [
    'python', 'train.py',
    '--data', OUT_DIR,
    '--epochs', '40',
    '--batch', '32',
    '--lr', '0.0001',
    '--out', '/kaggle/working/best_model.pth',
    '--workers', '4',
]
if HF_TOKEN:
    args += ['--hf_repo', HF_REPO, '--hf_token', HF_TOKEN]

print('Запускаю обучение...')
result = subprocess.run(args, capture_output=False, text=True)
if result.returncode != 0:
    raise RuntimeError(f'❌ train.py завершился с кодом {result.returncode}')

In [ ]:
# ── Шаг 9: Проверка модели ────────────────────────────────────────────────────
import json

with open('class_map.json') as f:
    cm = json.load(f)
print('class_map.json:', json.dumps(cm, indent=2, ensure_ascii=False))

import os
size_mb = os.path.getsize('/kaggle/working/best_model.pth') / 1024 / 1024
print(f'\nРазмер модели: {size_mb:.1f} MB')

## После обучения

1. Скачайте `best_model.pth` и `class_map.json` (Output → кнопка Download)
2. Загрузите их на HuggingFace: `danyil163/SCINCOACH` (скрипт делает это автоматически если есть HF_TOKEN)
3. Railway автоматически подхватит новую модель при следующем cold start

**Целевые метрики:**
- val_acc > 80% — хорошо
- val_acc > 85% — отлично  
- val_acc > 70% для редких классов (vitiligo, rosacea) — приемлемо
